# Portfolio Data Analyst: Marketing Funnel & Channel Performance
**Dataset:** Olist E-Commerce Marketing Funnel (Kaggle)  
**Tujuan Bisnis:**
1. Mengidentifikasi channel mana yang konversinya paling tinggi dan mana yang sales cycle-nya paling cepat.
2. Memberikan rekomendasi alokasi budget berbasis efisiensi konversi dan kecepatan closing.

Struktur proyek ini terbagi menjadi **4 LANGKAH UTAMA**:
1. 📂 **LANGKAH 1 — AMBIL & BERSIHIN DATA** (Baca CSV, Tangani Data Kosong, Rapihin Format Tanggal)
2. 📊 **LANGKAH 2 — ANALISIS DATA & SQL** (Hitung Rata-rata, Cari Tren, Bandingin Data & Uji Statistik)
3. 🎨 **LANGKAH 3 — VISUALISASI DATA** (Grafik Line, Bar Chart dengan Matplotlib & Seaborn)
4. ⚙️ **LANGKAH 4 — OTOMATISASI LAPORAN** (Multi-sheet Excel & CSV Final untuk Tableau)


---
## LANGKAH 1: AMBIL & BERSIHIN DATA
**Prinsip Data Analyst:** *Data mentah ≠ Data siap pakai.*  
Di langkah ini kita membaca file CSV mentah, memeriksa kualitas data, menangani missing values, merapikan format tanggal, dan menggabungkan tabel MQL dan Deals menjadi tabel funnel utuh.


### 1.1 Baca File CSV (Load Data)
Mendeteksi path dataset otomatis (kompatibel dengan Kaggle `/kaggle/input` maupun environment lokal).


In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats

# Path spesifik dataset di Kaggle
kaggle_path = '/kaggle/input/datasets/nofaamanda/marketing-funnel-by-olist'

def get_data_path(filename):
    # 1. Cek path spesifik dataset Kaggle kamu
    target_kaggle = os.path.join(kaggle_path, filename)
    if os.path.exists(target_kaggle):
        return target_kaggle
    
    # 2. Cek direktori /kaggle/input secara rekursif
    if os.path.exists('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            if filename in files:
                return os.path.join(root, filename)
                
    # 3. Fallback lingkungan lokal
    for local_dir in ['.', 'data/raw']:
        local_target = os.path.join(local_dir, filename)
        if os.path.exists(local_target):
            return local_target
            
    return filename

mql_path = get_data_path('olist_marketing_qualified_leads_dataset.csv')
deals_path = get_data_path('olist_closed_deals_dataset.csv')

print(f"Path MQL   : {mql_path}")
print(f"Path Deals : {deals_path}")

df_mql_raw = pd.read_csv(mql_path)
df_deals_raw = pd.read_csv(deals_path)

print(f"\n[OK] MQL raw loaded: {df_mql_raw.shape[0]:,} baris, {df_mql_raw.shape[1]} kolom")
print(f"[OK] Closed Deals raw loaded: {df_deals_raw.shape[0]:,} baris, {df_deals_raw.shape[1]} kolom")


### 1.2 Hapus & Tangani Data Kosong (Handling Missing & Cleaning)
- Memeriksa keunikan `mql_id` (Primary Key).
- Mengimputasi 60 baris kosong pada `origin` menjadi `'unknown'` agar lead tidak hilang saat diagregasi.
- Mengimputasi missing values pada segmen bisnis deals.


In [ ]:
# 1. Cek Duplikasi Primary Key
print("=== 1. CEK DUPLIKASI mql_id ===")
print(f"Duplikat di MQL   : {df_mql_raw['mql_id'].duplicated().sum()}")
print(f"Duplikat di Deals : {df_deals_raw['mql_id'].duplicated().sum()}")

# 2. Salin data untuk dibersihkan
mql_clean = df_mql_raw.copy()
deals_clean = df_deals_raw.copy()

# 3. Imputasi data kosong (Missing Values)
mql_clean['origin'] = mql_clean['origin'].fillna('unknown')
deals_clean['business_segment'] = deals_clean['business_segment'].fillna('unknown')
deals_clean['lead_type'] = deals_clean['lead_type'].fillna('unknown')
deals_clean['business_type'] = deals_clean['business_type'].fillna('unknown')

print("\n[OK] Data kosong pada origin dan segmen bisnis berhasil diimputasi.")
print("Total origin 'unknown' sekarang:", (mql_clean['origin'] == 'unknown').sum())


### 1.3 Rapihin Format Data & Feature Engineering
- Parsing format tanggal ke `datetime`.
- Ekstrak cohort bulan first contact (`lead_month`).
- Left Join MQL ke Closed Deals (Funnel utuh).
- Buat fitur: `is_won` (1/0) dan durasi penutupan transaksi `sales_cycle_days` (koreksi anomali negatif ke 0).
- Audit keterbatasan `declared_monthly_revenue` (94,66% bernilai 0).


In [ ]:
# 1. Parsing Tanggal
mql_clean['first_contact_date'] = pd.to_datetime(mql_clean['first_contact_date'])
deals_clean['won_date'] = pd.to_datetime(deals_clean['won_date'])

# 2. Ekstrak Periode Bulan
mql_clean['lead_month'] = mql_clean['first_contact_date'].dt.to_period('M').astype(str)

# 3. Left Join: SEMUA Leads digabung ke Deals
df_funnel = pd.merge(mql_clean, deals_clean, on='mql_id', how='left')

# 4. Feature Engineering
df_funnel['is_won'] = df_funnel['won_date'].notnull().astype(int)
df_funnel['sales_cycle_days'] = (df_funnel['won_date'] - df_funnel['first_contact_date']).dt.days
# Koreksi anomali tanggal: jika sales_cycle < 0 (data entry timing anomaly), set ke 0
df_funnel['sales_cycle_days'] = df_funnel['sales_cycle_days'].apply(lambda x: 0 if x < 0 else x)

# 5. Audit Keterbatasan Data Revenue
won_deals = df_funnel[df_funnel['is_won'] == 1]
zeros_rev = (won_deals['declared_monthly_revenue'] == 0).sum()
print("=== AUDIT KETERBATASAN DATA REVENUE ===")
print(f"Total Won Deals              : {len(won_deals):,}")
print(f"Revenue bernilai 0           : {zeros_rev:,} ({zeros_rev / len(won_deals) * 100:.2f}%)")
print("Catatan Metodologi: Alokasi budget difokuskan pada Conversion Rate & Sales Cycle Days, bukan total revenue.")

# 6. Simpan File Bersih
os.makedirs('data/clean', exist_ok=True)
df_funnel.to_csv('data/clean/funnel_clean.csv', index=False)
df_funnel.to_csv('funnel_clean.csv', index=False)
print("\n[OK] Langkah 1 Selesai: funnel_clean.csv tersimpan!")


---
## LANGKAH 2: ANALISIS DATA (Python & SQL)
Pada langkah ini, kita menggunakan **SQL (SQLite)** dan **Python** untuk:
1. **Hitung rata-rata**: Conversion rate dan durasi sales cycle per marketing channel.
2. **Cari tren**: Tren cohort bulanan (volume leads vs closing).
3. **Bandingin data**: Performa landing page (min. 30 leads) dan segmen bisnis.
4. **Uji Signifikansi Statistik**: Uji Chi-Square untuk memastikan apakah perbedaan konversi antar channel nyata atau sekadar noise sampel kecil.


### 2.1 Load Data ke SQLite & Analisis Channel (Hitung Rata-rata)
Menghitung Conversion Rate (%) dan Rata-rata Sales Cycle (Hari) per Marketing Channel menggunakan SQL.


In [ ]:
# 1. Inisialisasi Database SQLite in-memory dari tabel funnel_clean
conn = sqlite3.connect(':memory:')
df_funnel.to_sql('funnel', conn, index=False, if_exists='replace')

# 2. Query SQL: Performa Conversion Rate & Sales Cycle per Origin (Channel)
sql_channel = """
SELECT 
    origin AS marketing_channel,
    COUNT(mql_id) AS total_leads,
    SUM(is_won) AS won_deals,
    ROUND(CAST(SUM(is_won) AS FLOAT) / COUNT(mql_id) * 100, 2) AS conv_rate_pct,
    ROUND(AVG(CASE WHEN is_won = 1 THEN sales_cycle_days END), 1) AS avg_sales_cycle_days
FROM funnel
GROUP BY origin
ORDER BY conv_rate_pct DESC
"""

df_channel_sql = pd.read_sql(sql_channel, conn)
print("=== HASIL QUERY SQL: PERFORMA CHANNEL ===")
display(df_channel_sql)

# 3. Validasi & Komparasi dengan Pandas Groupby (Pembuktian Konsistensi)
df_channel_pandas = df_funnel.groupby('origin').agg(
    total_leads=('mql_id', 'count'),
    won_deals=('is_won', 'sum'),
    avg_sales_cycle_days=('sales_cycle_days', 'mean')
).reset_index()
df_channel_pandas['conv_rate_pct'] = round(df_channel_pandas['won_deals'] / df_channel_pandas['total_leads'] * 100, 2)
df_channel_pandas['avg_sales_cycle_days'] = round(df_channel_pandas['avg_sales_cycle_days'], 1)
df_channel_pandas = df_channel_pandas.sort_values('conv_rate_pct', ascending=False).reset_index(drop=True)

print("\n[VALIDASI] Perbandingan SQL vs Pandas: Apakah hasilnya 100% identik?")
is_match = df_channel_sql['conv_rate_pct'].equals(df_channel_pandas['conv_rate_pct'])
print(f"Status Pencocokan Hasil: {'COCOK / IDENTIK' if is_match else 'BERBEDA'}")


### 2.2 Cari Tren Bulanan (Monthly Cohort) & Bandingin Landing Page
- Menganalisis tren pergerakan lead vs won per bulan.
- Membandingkan performa Landing Page (disaring dengan filter minimal 30 lead untuk menghindari bias sampel kecil).


In [ ]:
# 1. Query SQL: Tren Bulanan (Monthly Funnel Cohort)
sql_monthly = """
SELECT 
    lead_month,
    COUNT(mql_id) AS total_leads,
    SUM(is_won) AS won_deals,
    ROUND(CAST(SUM(is_won) AS FLOAT) / COUNT(mql_id) * 100, 2) AS conv_rate_pct
FROM funnel
GROUP BY lead_month
ORDER BY lead_month ASC
"""
df_monthly_trend = pd.read_sql(sql_monthly, conn)
print("=== 1. TREN FUNNEL BULANAN (SQL) ===")
display(df_monthly_trend)

# 2. Query SQL: Performa Landing Page (Minimal 30 leads)
sql_landing = """
SELECT 
    landing_page_id,
    COUNT(mql_id) AS total_leads,
    SUM(is_won) AS won_deals,
    ROUND(CAST(SUM(is_won) AS FLOAT) / COUNT(mql_id) * 100, 2) AS conv_rate_pct
FROM funnel
GROUP BY landing_page_id
HAVING total_leads >= 30
ORDER BY conv_rate_pct DESC
LIMIT 10
"""
df_landing_perf = pd.read_sql(sql_landing, conn)
print("\n=== 2. TOP 10 LANDING PAGES (Min. 30 Leads) (SQL) ===")
display(df_landing_perf)

# 3. Query SQL: Breakdown Segmen & Tipe Bisnis (Closed Deals)
sql_segment = """
SELECT 
    COALESCE(business_segment, 'unknown') AS business_segment,
    COALESCE(business_type, 'unknown') AS business_type,
    COUNT(mql_id) AS total_won_deals,
    ROUND(COUNT(mql_id) * 100.0 / (SELECT COUNT(*) FROM funnel WHERE is_won = 1), 2) AS pct_of_total_deals,
    ROUND(AVG(sales_cycle_days), 1) AS avg_sales_cycle_days
FROM funnel
WHERE is_won = 1
GROUP BY business_segment, business_type
ORDER BY total_won_deals DESC
LIMIT 10
"""
df_segment_perf = pd.read_sql(sql_segment, conn)
print("\n=== 3. BREAKDOWN SEGMEN & TIPE BISNIS (SQL) ===")
display(df_segment_perf)


### 2.3 Uji Signifikansi Statistik (Chi-Square Test)
Apakah perbedaan rasio konversi antar channel benar-benar signifikan secara statistik, atau hanya fluktuasi acak karena ukuran sampel channel kecil?


In [ ]:
# Tabel Kontingensi (Origin vs is_won)
contingency_table = pd.crosstab(df_funnel['origin'], df_funnel['is_won'])

chi2_stat, p_val, dof, expected = stats.chi2_contingency(contingency_table)

print("=== HASIL UJI HIPOTESIS (CHI-SQUARE TEST) ===")
print(f"Chi-Square Statistic : {chi2_stat:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"P-Value              : {p_val:.4e}")

alpha = 0.05
if p_val < alpha:
    print(f"\nKesimpulan Statistik: p-value < {alpha} -> Perbedaan konversi antar channel SIGNIFIKAN secara statistik (Bukan kebetulan/noise).")
else:
    print(f"\nKesimpulan Statistik: p-value >= {alpha} -> Tidak cukup bukti statistik.")


---
## LANGKAH 3: VISUALISASI DATA (Matplotlib & Seaborn)
Sesuai prinsip visualisasi data:
- **Grafik Line**: Memvisualisasikan tren volume leads vs closing per bulan.
- **Bar Chart**: Menyorot perbandingan Conversion Rate dan rata-rata Sales Cycle per channel.
- Menggunakan palet profesional (Navy & Accent Orange). Gambar disimpan otomatis ke folder `reports/`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('reports', exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# -------------------------------------------------------------
# Chart 1: Line Chart Tren Bulanan Lead vs Won
# -------------------------------------------------------------
fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

months = df_monthly_trend['lead_month']
leads = df_monthly_trend['total_leads']
wons = df_monthly_trend['won_deals']
convs = df_monthly_trend['conv_rate_pct']

ax1.plot(months, leads, marker='o', color='#1A365D', linewidth=2.5, label='Total Leads (MQL)')
ax1.plot(months, wons, marker='s', color='#2B6CB0', linewidth=2, label='Won Deals')
ax2.plot(months, convs, marker='^', color='#DD6B20', linewidth=2, linestyle='--', label='Conversion Rate (%)')

ax1.set_title('Tren Bulanan Marketing Funnel (Leads vs Won Deals)', fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel('Cohort Bulan First Contact', fontsize=11)
ax1.set_ylabel('Volume Leads & Deals', fontsize=11, color='#1A365D')
ax2.set_ylabel('Conversion Rate (%)', fontsize=11, color='#DD6B20')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, linestyle=':', alpha=0.6)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

plt.tight_layout()
plt.savefig('reports/trend_monthly_funnel.png', dpi=300)
plt.show()

# -------------------------------------------------------------
# Chart 2: Bar Chart Conversion Rate per Channel
# -------------------------------------------------------------
plt.figure(figsize=(10, 5))
sorted_conv = df_channel_perf.sort_values('conv_rate_pct', ascending=False)
colors_conv = ['#DD6B20' if i < 2 else '#2B6CB0' for i in range(len(sorted_conv))]

ax = sns.barplot(data=sorted_conv, x='marketing_channel', y='conv_rate_pct', hue='marketing_channel', palette=colors_conv, legend=False)
plt.title('Conversion Rate (%) per Marketing Channel', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Marketing Channel', fontsize=11)
plt.ylabel('Conversion Rate (%)', fontsize=11)
plt.xticks(rotation=35, ha='right')

for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('reports/channel_conversion_rate.png', dpi=300)
plt.show()

# -------------------------------------------------------------
# Chart 3: Bar Chart Sales Cycle Days per Channel
# -------------------------------------------------------------
plt.figure(figsize=(10, 5))
sorted_cycle = df_channel_perf.sort_values('avg_sales_cycle_days', ascending=True)
colors_cycle = ['#38A169' if i < 2 else '#4A5568' for i in range(len(sorted_cycle))]

ax = sns.barplot(data=sorted_cycle, x='marketing_channel', y='avg_sales_cycle_days', hue='marketing_channel', palette=colors_cycle, legend=False)
plt.title('Rata-rata Sales Cycle (Hari) per Channel (Lebih Cepat = Lebih Baik)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Marketing Channel', fontsize=11)
plt.ylabel('Rata-rata Durasi Closing (Hari)', fontsize=11)
plt.xticks(rotation=35, ha='right')

for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}h", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('reports/channel_sales_cycle.png', dpi=300)
plt.show()

print("[OK] Semua visualisasi berhasil disimpan di folder 'reports/'!")


---
## LANGKAH 4: OTOMATISASI LAPORAN
**Nilai Tambah:** Membangun pipeline otomatisasi pelaporan agar menghemat waktu operasional:
1. Menghasilkan laporan Excel multi-sheet otomatis (`reports/marketing_report.xlsx`).
2. Menghasilkan file dataset final (`data/clean/tableau_funnel.csv`) yang terstruktur sempurna untuk dashboard Tableau Public.


In [ ]:
# 1. Pipeline Otomatisasi Export ke Excel Multi-Sheet
report_excel_path = 'reports/marketing_report.xlsx'

try:
    with pd.ExcelWriter(report_excel_path, engine='openpyxl') as writer:
        df_channel_perf.to_excel(writer, sheet_name='Channel_Performance', index=False)
        df_monthly_trend.to_excel(writer, sheet_name='Monthly_Trend', index=False)
        df_landing_perf.to_excel(writer, sheet_name='Landing_Page_Performance', index=False)
    print(f"[SUKSES] Laporan Excel dibuat: {report_excel_path}")
except Exception as e:
    # Fallback jika openpyxl belum terinstall
    df_channel_perf.to_csv('reports/report_channel_perf.csv', index=False)
    df_monthly_trend.to_csv('reports/report_monthly_trend.csv', index=False)
    df_landing_perf.to_csv('reports/report_landing_perf.csv', index=False)
    print(f"[INFO] CSV Fallback dibuat di folder 'reports/' (karena error: {e})")

# 2. Export CSV Siap Pakai untuk Dashboard Tableau Public
tableau_csv_path = 'data/clean/tableau_funnel.csv'
df_funnel.to_csv(tableau_csv_path, index=False)
df_funnel.to_csv('tableau_funnel.csv', index=False)

print(f"[SUKSES] Dataset final Tableau siap pakai disimpan di: {tableau_csv_path}")


---
## KESIMPULAN & JAWABAN PROBLEM STATEMENT

### 1. Channel Konversi Tertinggi & Sales Cycle Tercepat:
- **Konversi Tertinggi (Kecuali unknown)**: **Paid Search (12,30%)** dan **Organic Search (11,80%)**. Keduanya menyumbang volume deal terbanyak (195 dan 271 deals).
- **Sales Cycle Tercepat**: **Display (10,3 hari)** dan **Direct Traffic (31,1 hari)**. 
- **Channel Paling Lemah**: **Email (3,04%)** dan **Social (5,56%)** dengan sales cycle sangat lambat (61 hari).

### 2. Rekomendasi Alokasi Budget Marketing:
- **Tingkatkan Alokasi ke Paid Search**: Terbukti memiliki konversi tertinggi di antara channel berbayar (12,30%) dengan volume leads stabil.
- **Investasi di SEO (Organic Search)**: Memiliki konversi tinggi (11,80%) tanpa beban biaya per-klik langsung.
- **Evaluasi Ulang / Pangkas Budget Social Media**: Social menghasilkan 1.350 lead namun konversinya hanya 5,56% dan durasi closing sangat lama (61 hari).
- **Catatan Keterbatasan Data**: Keputusan budget dibuat berdasarkan rasio konversi dan kecepatan closing, bukan total `declared_monthly_revenue` karena 94,66% data revenue adalah self-reported 0.
